# 2026-11-23 EDA on FOODS, day 3
Messy exploring is allowed here. Anything worth keeping moves to the EDA doc.

## Open from earlier days
- **Alive:** how do we define when an item is "alive" in a store? (from 20 Nov)
- **Wisconsin stores 1 and 2:** handle the level jump how? Train after the jump, or add a level flag? (21 Nov)
- **Holidays:** no smooth yearly season, but do Christmas, Thanksgiving etc. matter as one-off events? (21 Nov)
- **Stationarity:** do we need it at all? Depends on ARIMA-style vs tree-based baseline. (21 Nov)

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from m5food import data, plots
plots.set_style()

## Load
Plain names (`state`, `store`, `dept`) are added next to the ids. Use the plain names on every chart.

In [ ]:
sales = data.add_readable_names(data.load_sales_foods())
cal = data.load_calendar()
days = data.day_columns(sales)
dates = cal.set_index('d').loc[days, 'date'].values      # d_1 ... d_1941 -> dates
sales.shape

## Carried over: per-series helpers
Zero days are counted only from each series' first sale.

In [ ]:
X = sales[days].to_numpy()
first = (X > 0).argmax(axis=1)                        # first day with a sale
live = np.arange(X.shape[1]) >= first[:, None]        # days from the first sale on
sales['first_sale'] = pd.to_datetime(dates[first])
sales['zero_share'] = ((X == 0) & live).sum(1) / live.sum(1) * 100
sales['mean_daily'] = np.nanmean(np.where(live, X, np.nan), axis=1)
sales[['item_id', 'store', 'dept', 'first_sale', 'zero_share', 'mean_daily']].head()

## Carried over: daily totals and clean-up helpers
Christmas closures filled, straight-line trend, weekday pattern.

In [ ]:
from statsmodels.tsa.stattools import acf

total = pd.Series(X.sum(axis=0), index=pd.DatetimeIndex(dates), name='All FOODS')   # units per day, all stores
noise = 1.96 / np.sqrt(len(total))                                                   # ACF noise band

from plotly.subplots import make_subplots

# Clean-up helpers used by the charts below
def fill_christmas(s):
    """Stores are closed on 25 Dec (about 15 units instead of ~30,000). Use the average of the day before and after."""
    s = s.astype(float).copy()
    for d in s.index[(s.index.month == 12) & (s.index.day == 25)]:
        s[d] = (s[d - pd.Timedelta(days=1)] + s[d + pd.Timedelta(days=1)]) / 2
    return s

def remove_trend(s):
    """Subtract a straight-line trend fitted over the whole period."""
    t = np.arange(len(s))
    slope, intercept = np.polyfit(t, s.to_numpy(dtype=float), 1)
    return s - (slope * t + intercept)

def remove_weekday(s):
    """Subtract the average of each weekday, so the weekly pattern is gone and other cycles can show."""
    return s - s.groupby(s.index.dayofweek).transform('mean')

by_store_day = sales.groupby('store')[days].sum().T; by_store_day.index = total.index
by_state_day = sales.groupby('state')[days].sum().T; by_state_day.index = total.index
STATE_COLOR = {'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}

## Start of the story: one store at a time
Before any averages: what does a single store's FOODS sales look like?
For each state: the **best**, a **typical** and the **weakest** store, by average FOODS units sold per day over the whole period.

In [ ]:
# Story opener, store level: best, typical and weakest store in each state, by average FOODS units sold per day.
# SHOW_DAYS = how many of the last days to draw: 56 = last 8 weeks (bars), 1941 = all 5+ years (dark lines).
SHOW_DAYS = 1941
STATES = ['California', 'Texas', 'Wisconsin']
KINDS = ['Best store', 'Typical store', 'Weakest store']
KIND_COLOR = {'Best store': plots.PALETTE[0], 'Typical store': plots.PALETTE[4], 'Weakest store': plots.PALETTE[1]}
store_avg = by_store_day.mean()                                  # average units per day, whole period

picks = []
for st in STATES:
    avg = store_avg[[s for s in store_avg.index if s.startswith(st)]].sort_values()
    typical = (avg - avg.median()).abs().idxmin()                # store closest to the state's median
    for kind, store in zip(KINDS, [avg.idxmax(), typical, avg.idxmin()]):
        y = by_store_day[store]
        picks.append({'state': st, 'kind': kind, 'store': store, 'units a day': y.mean(),
                      'first year a day': y.iloc[:365].mean(), 'last year a day': y.iloc[-365:].mean()})

win_dates = by_store_day.index[-SHOW_DAYS:]
fig = make_subplots(rows=3, cols=3, shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.1, horizontal_spacing=0.04,
                    subplot_titles=[f"<b>{p['state']}: {p['kind'].lower()}</b><br>"
                                    f"<span style='font-size:11px'>{p['store']} · {p['units a day']:,.0f} a day · "
                                    f"first year {p['first year a day']:,.0f} → last year {p['last year a day']:,.0f}</span>"
                                    for p in picks])
for k, p in enumerate(picks):
    y = by_store_day[p['store']].iloc[-SHOW_DAYS:]
    hover = f"{p['store']}<br>%{{x|%a %d %b %Y}}: %{{y:,}} sold<extra></extra>"
    if SHOW_DAYS <= 120:     # few days: one bar per day
        trace = go.Bar(x=win_dates, y=y, marker=dict(color=KIND_COLOR[p['kind']], line_width=0), hovertemplate=hover)
    else:                    # many days: bars become hairlines and look pale, so draw a solid line instead
        trace = go.Scatter(x=win_dates, y=y, mode='lines', line=dict(color=KIND_COLOR[p['kind']], width=1),
                           hovertemplate=hover)
    fig.add_trace(trace.update(showlegend=False), row=k // 3 + 1, col=k % 3 + 1)
fig.update_yaxes(rangemode='tozero')
fig.update_yaxes(title_text='FOODS units sold', col=1)
fig.update_layout(title='<b>One store at a time: every store sells food every day, but at very different levels</b><br>'
                        f'<span style="font-size:13px;color:#444">{win_dates[0]:%d %b %Y} to {win_dates[-1]:%d %b %Y}, '
                        'one value per day, same scale in every panel. The yearly drops to near 0 are Christmas Day (stores closed).</span>',
                  template='simple_white', height=820, bargap=0.1, margin=dict(t=150))
fig.show()
pd.DataFrame(picks).round(0)

**Draft observations (check against the chart)**
- **Every store sells food every day.** The only drops to near 0 are Christmas Day, when stores are closed. So the 40-60% no-sale days are an *item* problem, not a store problem.
- **Big gaps inside a state.** California's best store (store 3, about 3,900 a day) sells about 2.7 times its weakest (store 4, about 1,500). Texas stores sit close together (about 2,000 to 2,600) and grew the least.
- **Wisconsin's ranking changed over time.** Wisconsin stores 1 and 2 started as the smallest (about 900 and 1,200 a day) and roughly tripled (to about 2,500 and 3,300); Wisconsin store 3 stayed flat (about 2,500). This is the level jump from day 2. California store 2 also jumped (1,500 to 2,600).
- **So what:** the store level looks healthy and smooth. To understand the empty days we have to go down to single products.

### One product at a time
Same idea, now per product, whatever the department or store: each item's sales in **all 10 stores added together**.
**Best** = highest average units per day since the item's first sale, **typical** = closest to the median item, **weakest** = lowest.
Only items that started before 2015 and still sold in the last 8 weeks (1,419 of 1,437 items).

In [ ]:
# Story, product level: ONE best, ONE typical and ONE weakest FOODS item, all 10 stores added together.
# Average per day is counted from the item's first sale in any store.
# Only items that started before 2015 and still sold in the last 8 weeks, so new or dead items do not distort it.
SHOW_DAYS = 1941
item_day = sales.groupby('item_id')[days].sum()                      # one row per item, all stores added
I = item_day.to_numpy()
i_first = (I > 0).argmax(axis=1)
i_live = np.arange(I.shape[1]) >= i_first[:, None]
items = pd.DataFrame({'item': item_day.index,
                      'dept': item_day.index.str.rsplit('_', n=1).str[0].map(data.dept_name),
                      'first sale': pd.to_datetime(dates[i_first]),
                      'units a day': np.nanmean(np.where(i_live, I, np.nan), axis=1),
                      'no-sale days %': ((I == 0) & i_live).sum(1) / i_live.sum(1) * 100})
pool = items[(items['first sale'] < '2015-01-01') & (I[:, -56:].sum(1) > 0)]

KINDS = ['Best item', 'Typical item', 'Weakest item']
KIND_COLOR = {'Best item': plots.PALETTE[0], 'Typical item': plots.PALETTE[4], 'Weakest item': plots.PALETTE[1]}
u = pool['units a day']
item_picks = [pool.loc[u.idxmax()], pool.loc[(u - u.median()).abs().idxmin()], pool.loc[u.idxmin()]]

win_dates = pd.DatetimeIndex(dates[-SHOW_DAYS:])
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.06,
                    subplot_titles=[f"<b>{kind}</b><br><span style='font-size:11px'>{r['dept']}, item "
                                    f"{r['item'].split('_')[-1]} · {r['units a day']:.1f} a day · "
                                    f"no sale on {r['no-sale days %']:.0f}% of days</span>"
                                    for kind, r in zip(KINDS, item_picks)])
for j, (kind, r) in enumerate(zip(KINDS, item_picks), start=1):
    y = I[items.index[items.item == r['item']][0], -SHOW_DAYS:]
    hover = f"{r['dept']}, item {r['item'].split('_')[-1]}<br>%{{x|%a %d %b %Y}}: %{{y}} sold (all stores)<extra></extra>"
    if SHOW_DAYS <= 120:
        trace = go.Bar(x=win_dates, y=y, marker=dict(color=KIND_COLOR[kind], line_width=0), hovertemplate=hover)
    else:
        trace = go.Scatter(x=win_dates, y=y, mode='lines', line=dict(color=KIND_COLOR[kind], width=1), hovertemplate=hover)
    fig.add_trace(trace.update(showlegend=False), row=1, col=j)
fig.update_yaxes(rangemode='tozero')
fig.update_yaxes(title_text='units sold, all 10 stores', col=1)
fig.update_layout(title='<b>One product at a time: the best item sells every day, a typical item often sells nothing</b><br>'
                        f'<span style="font-size:13px;color:#444">{win_dates[0]:%d %b %Y} to {win_dates[-1]:%d %b %Y}, '
                        'one value per day, all 10 stores added together. Each panel has its own scale.</span>',
                  template='simple_white', height=460, bargap=0.1, margin=dict(t=140))
fig.show()
pd.DataFrame(item_picks, index=KINDS).round(1)

**Draft observations (check against the chart)**
- **Best item: Food dept 3, item 090, about 524 a day across all stores.** Yet it sold nothing *anywhere* on 17% of its days (329 days). A product this popular does not lose all demand in 10 stores at once, so those stretches look like the item was unavailable (stock-out or taken off the shelf). This is a strong Phase 2 example.
- **Typical item: Food dept 2, item 146, about 10 a day across 10 stores** (about 1 per store per day). Added up over all stores it is empty on only 8% of days, even though one store alone would often sell nothing.
- **Weakest item: Food dept 1, item 079, under 1 a day across all 10 stores** (1,734 units in 5 years). It sells nothing on 62% of days even with every store added together.
- **So what:** adding stores together hides most empty days for typical items. The empty days live at the single item-in-one-store level, which is the level we have to forecast.

## Today

## Plan: are the zero days seasonal, or something else?
**Phase 1 (seasonality).** Hypothesis: many zero days come from seasonal items that only sell in some months (like diyas at Diwali). If true, 40-50% zero days is expected and fine.
Each step is a gate: go one level down only if the level above shows something.
1. **State** × calendar month: zero share. *(today, below)*
2. **Store** × calendar month.
3. **Food dept 1 / 2 / 3** × calendar month.
4. **Items** in the department that stands out: which items sell only in some months?

**Phase 2 (availability vs demand).** Are zero days caused by stock-outs (demand there, no stock) or by no demand?

### Step 1: zero share by calendar month, per state
Zero days counted only from each item's first sale. 25 Dec is left out because stores are closed.
Only full years **2012-2015** go into the month effect: 2011 starts on 29 Jan and 2016 stops in May, so they would tilt Jan-May.
**Month effect** = that month's zero share minus the same year's average, averaged over 2012-2015. It removes the slow drift between years.

In [ ]:
# Zero-day share by calendar month, per state.
# - Counted only from each item's first sale (live days).
# - 25 Dec left out: stores are closed, so every item is 0 that day.
# - Only full years 2012-2015 in the average. 2011 starts on 29 Jan and 2016 stops in May, so they would
#   give Jan-May one extra (odd) year each and tilt the months.
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
FULL_YEARS = [2012, 2013, 2014, 2015]
day_idx = pd.DatetimeIndex(dates)
open_day = ~((day_idx.month == 12) & (day_idx.day == 25))
zero_live = (X == 0) & live                                  # True = a live day with no sale

def zero_by_year_month(row_mask):
    """Zero share (%) for the chosen rows: one row per year, one column per month."""
    d = pd.DataFrame({'zeros': zero_live[row_mask].sum(0), 'live': live[row_mask].sum(0),
                      'year': day_idx.year, 'month': day_idx.month})[open_day]
    g = d.groupby(['year', 'month'])[['zeros', 'live']].sum()
    out = (g.zeros / g.live * 100).unstack('month')
    out.columns = MONTHS
    return out

def month_pattern(row_mask):
    """For each full year: month's zero share minus that year's average. Then average over the years.
    Removes the slow year-to-year drift, so only the calendar-month effect is left (in % points)."""
    ym = zero_by_year_month(row_mask).loc[FULL_YEARS]
    return ym.sub(ym.mean(axis=1), axis=0).mean()

STATES = ['California', 'Texas', 'Wisconsin']
state_ym = {st: zero_by_year_month((sales.state == st).to_numpy()) for st in STATES}
state_pattern = pd.DataFrame({st: month_pattern((sales.state == st).to_numpy()) for st in STATES})
state_pattern.round(1).T          # + = more zero days than that year's average, − = fewer

In [ ]:
# Chart. Top: raw zero share, one line per year. Bottom: month effect with the yearly drift removed (2012-2015).
fig = make_subplots(rows=2, cols=3, shared_yaxes='rows', subplot_titles=STATES + [''] * 3,
                    vertical_spacing=0.14, horizontal_spacing=0.04, row_heights=[0.55, 0.45])
for j, st in enumerate(STATES, start=1):
    ym = state_ym[st]
    for k, (year, row) in enumerate(ym.iterrows()):
        full = year in FULL_YEARS
        fig.add_trace(go.Scatter(x=list(row.dropna().index), y=row.dropna(), mode='lines',
                                 line=dict(color=STATE_COLOR[st] if full else '#aaa', width=1.5 if full else 1,
                                           dash='solid' if full else 'dot'),
                                 opacity=0.4 + 0.12 * k if full else 0.8, showlegend=False,
                                 hovertemplate=f'{st}, {year}<br>%{{x}}: %{{y:.1f}}% zero days<extra></extra>'),
                      row=1, col=j)
    pat = state_pattern[st]
    fig.add_trace(go.Bar(x=MONTHS, y=pat, marker_color=STATE_COLOR[st], showlegend=False,
                         text=pat.map('{:+.1f}'.format), textposition='outside',
                         hovertemplate=f'{st}<br>%{{x}}: %{{y:+.1f}} points vs year average<extra></extra>'),
                  row=2, col=j)
fig.update_yaxes(title_text='% zero days', row=1, col=1)
fig.update_yaxes(title_text='points vs year avg', range=[-4, 4], row=2, col=1)
fig.update_yaxes(range=[-4, 4], row=2)
fig.update_layout(title='<b>Do zero days pile up in certain months?</b><br>'
                        '<span style="font-size:13px;color:#444">Top: zero share per month, one line per year '
                        '(grey dotted = part years 2011 and 2016; darker = later year). '
                        'Bottom: month effect once the year-to-year drift is removed (2012-2015).</span>',
                  template='simple_white', height=720)
fig.show()
summary = pd.DataFrame({'2012 avg %': [state_ym[s].loc[2012].mean() for s in STATES],
                        '2015 avg %': [state_ym[s].loc[2015].mean() for s in STATES],
                        'month effect: biggest + (points)': state_pattern.max().values,
                        'month effect: biggest − (points)': state_pattern.min().values}, index=STATES)
summary.round(1)

**Draft observations (check against the chart)**
- **The month barely matters.** In every state each month stays within about **±3 points** of its year's average. The only visible bump is **Nov-Dec in California and Texas (+2 to +3 points)**, and February is the lowest month everywhere. Wisconsin is almost flat.
- **The year matters more than the month.** The zero share rises from 2012 to 2015 in every state: California 44% → 51%, Texas 50% → 58%, Wisconsin 48% → 53%. That drift is bigger than any month effect.
- **At state level, seasonality does not explain the 40-50% zero days.** This matches day 2: calendar month explained about 2% of daily sales. But seasonal items could still exist and cancel each other out in a state average. This chart cannot rule them out.
- *Trap seen:* Jan 2011 shows only about 16% zeros. That is not real: items count as live only from their first sale, and in the first weeks almost only items that just sold are live.

### The same finding, for a business audience

In [ ]:
# Same finding, for a business audience: two plain panels, the conclusion is the title.
# Left: out of every 100 days, how many had no sale, per calendar month (2012-2015 average). Full 0-100 scale on purpose.
# Right: the same number per year, to show what actually changes.
fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.1,
                    subplot_titles=['<b>Every month looks the same</b>', '<b>But it gets slightly high every year</b>'])
for k, st in enumerate(STATES):
    ym = state_ym[st].loc[FULL_YEARS]
    by_month, by_year = ym.mean(), ym.mean(axis=1)
    fig.add_trace(go.Scatter(x=MONTHS, y=by_month, mode='lines+markers', name=st, legendgroup=st,
                             line=dict(color=STATE_COLOR[st], width=3, dash=['solid', 'dash', 'dot'][k]),
                             hovertemplate=f'{st}, %{{x}}: %{{y:.0f}} of every 100 days had no sale<extra></extra>'),
                  row=1, col=1)
    fig.add_trace(go.Bar(x=[str(y) for y in FULL_YEARS], y=by_year, name=st, legendgroup=st, showlegend=False,
                         marker_color=STATE_COLOR[st], text=by_year.map('{:.0f}'.format), textposition='outside',
                         hovertemplate=f'{st}, %{{x}}: %{{y:.0f}} of every 100 days had no sale<extra></extra>'),
                  row=1, col=2)
fig.update_yaxes(range=[0, 100], title_text='days with no sale, out of every 100', row=1, col=1)
fig.update_yaxes(range=[0, 100], row=1, col=2)
fig.add_annotation(xref='x domain', yref='y domain', x=0.5, y=0.12, showarrow=False, row=1, col=1,
                   text='Quietest vs busiest month: only 4 to 5 days in 100 apart', font=dict(size=13, color='#444'))
fig.add_annotation(xref='x2 domain', yref='y2 domain', x=0.5, y=0.12, showarrow=False,
                   text='2012 to 2015: up 5 to 8 days in 100', font=dict(size=13, color='#444'))
fig.update_layout(title='<b>About half of all days an item sells nothing, and the time of year does not change that</b><br>'
                        '<span style="font-size:13px;color:#444">Food items in 10 stores, counted from the day each item '
                        'first sold. Average of 2012 to 2015.</span>',
                  template='simple_white', height=520, barmode='group',
                  legend=dict(orientation='h', y=-0.15, x=0.3))
fig.show()

**In plain words (draft)**
- On about **half of all days, a typical food item sells nothing** in a store. That is true in every state.
- **The time of year does not change this.** January and July look the same; the gap between the quietest and busiest month is only 4 to 5 days in 100.
- **What does change is the year:** from 2012 to 2014 the no-sale days rose by 5 to 8 in every 100, then levelled off in 2015.
- **So what:** we cannot explain the empty days as "seasonal products". Next we check whether they come from slow sellers or from items being out of stock.

### Step 2: zero share by calendar month, per store
Same rules as Step 1: live days only, 25 Dec left out, full years 2012-2015, yearly drift removed.
Each cell = how many points more (red) or fewer (blue) no-sale days that month has than the store's yearly average.

In [ ]:
# Step 2: zero share by calendar month, per store. Same rules as Step 1:
# live days only (from first sale), 25 Dec left out, month effect from full years 2012-2015 with the yearly drift removed.
STORES = sorted(sales.store.unique())                       # California 1-4, Texas 1-3, Wisconsin 1-3
store_ym = {s: zero_by_year_month((sales.store == s).to_numpy()) for s in STORES}
store_pattern = pd.DataFrame({s: month_pattern((sales.store == s).to_numpy()) for s in STORES}).T   # rows = stores

fig = go.Figure(go.Heatmap(z=store_pattern.to_numpy(), x=MONTHS, y=STORES,
                           text=store_pattern.map('{:+.1f}'.format).to_numpy(), texttemplate='%{text}',
                           colorscale='RdBu_r', zmid=0, zmin=-4, zmax=4, xgap=2, ygap=2,
                           colorbar=dict(title='points vs<br>year avg'),
                           hovertemplate='%{y}, %{x}: %{z:+.1f} points vs that year\'s average<extra></extra>'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(title='<b>Step 2: month effect on zero days, per store</b><br>'
                        '<span style="font-size:13px;color:#444">Red = more no-sale days than that year\'s average, '
                        'blue = fewer. Full years 2012-2015, yearly drift removed. Pale everywhere = no monthly season.</span>',
                  template='simple_white', height=520, margin=dict(t=110))
fig.show()

In [ ]:
# Per store: how big is the month effect compared with the change between years?
store_summary = pd.DataFrame({
    '2012 zero %': [store_ym[s].loc[2012].mean() for s in STORES],
    '2015 zero %': [store_ym[s].loc[2015].mean() for s in STORES],
    'month effect: most +': store_pattern.max(axis=1).values,
    'month effect: most −': store_pattern.min(axis=1).values,
    'worst month': store_pattern.idxmax(axis=1).values}, index=STORES)
store_summary['2012→2015 change'] = store_summary['2015 zero %'] - store_summary['2012 zero %']
store_summary.round(1)

**Draft observations (check against the chart)**
- **No store has a real monthly season.** Every cell is within ±4 points of the store's yearly average, the same as at state level.
- **All 7 California and Texas stores share one small shape:** November-December about +2 to +3 points, February lowest. It appears in every store, so it is real, but it is small. Wisconsin stores do not show it.
- **Wisconsin store 3 moves most within the year:** low in Jan-Mar (−3.8 in Feb), higher in Jul-Aug (+2). Still only about 6 points from lowest to highest month.
- **The change between years is bigger than any month effect:** +5 to +11 points from 2012 to 2015 in 8 stores. The exceptions are **Wisconsin stores 1 and 2 (about 0 to +1)**, the two stores whose sales roughly tripled.
- **Gate:** the store level does not change the Step 1 answer. Next: Step 3, the three food departments.

### Step 2, the same finding for a business audience

In [ ]:
# Same Step 2 finding for a business audience. One row per store, one dot per calendar month:
# out of every 100 days, how many had no sale (average of 2012-2015). Dots bunched together = the month does not matter.
fig = go.Figure()
STATE_OF = lambda s: s.split(' store ')[0]
rows = []
for s in STORES:
    by_month = store_ym[s].loc[FULL_YEARS].mean()                    # 12 values, raw no-sale days per 100
    lo, hi = by_month.idxmin(), by_month.idxmax()
    rows.append({'store': s, 'quietest month': lo, 'low': by_month[lo], 'worst month': hi, 'high': by_month[hi]})
    fig.add_trace(go.Scatter(x=[by_month[lo], by_month[hi]], y=[s, s], mode='lines', showlegend=False,
                             line=dict(color='#ccc', width=8), hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=by_month.values, y=[s] * 12, mode='markers', showlegend=False,
                             marker=dict(color=STATE_COLOR[STATE_OF(s)], size=9, line=dict(color='white', width=1)),
                             customdata=by_month.index,
                             hovertemplate=s + ', %{customdata}: %{x:.0f} of every 100 days had no sale<extra></extra>'))
    fig.add_annotation(x=by_month[hi], y=s, text=f'{hi} {by_month[hi]:.0f}', showarrow=False, xanchor='left',
                       xshift=8, font=dict(size=11, color='#444'))
    fig.add_annotation(x=by_month[lo], y=s, text=f'{lo} {by_month[lo]:.0f}', showarrow=False, xanchor='right',
                       xshift=-8, font=dict(size=11, color='#444'))
fig.update_xaxes(range=[0, 100], title_text='days with no sale, out of every 100 (each dot = one calendar month)')
fig.update_yaxes(autorange='reversed')
fig.update_layout(title='<b>In every store, all 12 months look almost the same</b><br>'
                        '<span style="font-size:13px;color:#444">Each dot is one month (average of 2012 to 2015). '
                        'Labels show the quietest and the worst month. The grey bar is the whole range: only 4 to 6 days in 100.</span>',
                  template='simple_white', height=560, margin=dict(t=110, l=150))
fig.show()
pd.DataFrame(rows).set_index('store').round(0)

**In plain words (draft)**
- In every store, a food item has **no sale on roughly 40 to 60 of every 100 days**, depending on the store.
- **The month hardly changes that.** In each store, the quietest and the worst month are only **4 to 6 days in 100 apart**.
- February is the quietest month in 8 of 10 stores; December (or November) is the worst in all California and Texas stores. Small, but the same everywhere.
- **So what:** the empty days are not explained by products that only sell in certain months. The big differences are *between stores*, not between months.

### Step 3: zero share by calendar month, per food department
Same rules as Steps 1 and 2. Top 3 rows: each department over all 10 stores. Below the line: each department inside each state, to check the shape repeats.

In [ ]:
# Step 3: zero share by calendar month, per food department. Same rules as Steps 1 and 2.
# Rows: each department over all stores, then each department inside each state (to check it is the same everywhere).
DEPTS = ['Food dept 1', 'Food dept 2', 'Food dept 3']
groups = {d: (sales.dept == d).to_numpy() for d in DEPTS}
for st in STATES:
    for d in DEPTS:
        groups[f'{d}, {st}'] = ((sales.dept == d) & (sales.state == st)).to_numpy()
dept_ym = {k: zero_by_year_month(m) for k, m in groups.items()}
dept_pattern = pd.DataFrame({k: month_pattern(m) for k, m in groups.items()}).T

fig = go.Figure(go.Heatmap(z=dept_pattern.to_numpy(), x=MONTHS, y=list(dept_pattern.index),
                           text=dept_pattern.map('{:+.1f}'.format).to_numpy(), texttemplate='%{text}',
                           colorscale='RdBu_r', zmid=0, zmin=-4, zmax=4, xgap=2, ygap=2,
                           colorbar=dict(title='points vs<br>year avg'),
                           hovertemplate='%{y}, %{x}: %{z:+.1f} points vs that year\'s average<extra></extra>'))
fig.add_hline(y=2.5, line=dict(color='#333', width=2))             # line under the 3 all-store rows
fig.update_yaxes(autorange='reversed')
fig.update_layout(title='<b>Step 3: month effect on zero days, per food department</b><br>'
                        '<span style="font-size:13px;color:#444">Top 3 rows: all stores. Below: each department in each state. '
                        'Red = more no-sale days than that year\'s average, blue = fewer. Full years 2012-2015.</span>',
                  template='simple_white', height=600, margin=dict(t=110, l=200))
fig.show()
dept_summary = pd.DataFrame({
    '2012 zero %': [dept_ym[k].loc[2012].mean() for k in dept_pattern.index],
    '2015 zero %': [dept_ym[k].loc[2015].mean() for k in dept_pattern.index],
    'month effect: most +': dept_pattern.max(axis=1).values,
    'month effect: most −': dept_pattern.min(axis=1).values,
    'worst month': dept_pattern.idxmax(axis=1).values,
    'quietest month': dept_pattern.idxmin(axis=1).values}, index=dept_pattern.index)
dept_summary.round(1)

**Draft observations (check against the chart)**
- **No department has a strong monthly season.** The biggest cell anywhere is +4.5 points (Food dept 1, California, November). Everything else is within about ±4.
- **But each department has its own small shape, and it repeats in every state:**
  - **Food dept 1:** a November peak (+2 to +4.5 points in each state) and a September low.
  - **Food dept 2:** more empty days in spring (April to June), fewer in January to March. The opposite of the other two.
  - **Food dept 3:** November-December peak in California and Texas (+2 to +4 points). This is the biggest department (823 of 1,437 items), so it is what made all California and Texas stores look higher in Nov-Dec in Step 2.
- **Food dept 3 has the fewest empty days (42% in 2012) but they rose the most (to 51% in 2015).** Food dept 1 and 2 stayed around 54-57%.
- **Gate:** no department stands out strongly. One caution: a department average can still hide items that only sell in some months, because one item's busy month cancels another item's quiet month. Only an item-level check can rule that out.

### Step 3, the same finding for a business audience

In [ ]:
# Step 3 for a business audience: one row per department, one dot per calendar month, 0-100 scale.
fig = go.Figure()
DEPT_COLOR = dict(zip(DEPTS, [plots.PALETTE[3], plots.PALETTE[0], plots.PALETTE[1]]))
for d in DEPTS:
    by_month = dept_ym[d].loc[FULL_YEARS].mean()
    lo, hi = by_month.idxmin(), by_month.idxmax()
    fig.add_trace(go.Scatter(x=[by_month[lo], by_month[hi]], y=[d, d], mode='lines', showlegend=False,
                             line=dict(color='#ccc', width=10), hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=by_month.values, y=[d] * 12, mode='markers', showlegend=False,
                             marker=dict(color=DEPT_COLOR[d], size=11, line=dict(color='white', width=1)),
                             customdata=by_month.index,
                             hovertemplate=d + ', %{customdata}: %{x:.0f} of every 100 days had no sale<extra></extra>'))
    for m, side, shift in [(hi, 'left', 10), (lo, 'right', -10)]:
        fig.add_annotation(x=by_month[m], y=d, text=f'{m} {by_month[m]:.0f}', showarrow=False,
                           xanchor=side, xshift=shift, font=dict(size=12, color='#444'))
fig.update_xaxes(range=[0, 100], title_text='days with no sale, out of every 100 (each dot = one calendar month)')
fig.update_yaxes(autorange='reversed')
fig.update_layout(title='<b>In every food department, all 12 months look almost the same</b><br>'
                        '<span style="font-size:13px;color:#444">Each dot is one month (average of 2012 to 2015), all 10 stores. '
                        'Labels show the quietest and the worst month.</span>',
                  template='simple_white', height=380, margin=dict(t=110, l=120))
fig.show()

**In plain words (draft)**
- In each food department, items have **no sale on about 46 to 60 of every 100 days**.
- **The month hardly changes it:** quietest vs worst month is only **4 to 6 days in 100** in each department.
- Food dept 3 (the biggest) has the fewest empty days; Food dept 1 and 2 have more.
- **So what:** no department behaves like a "Diwali diya" department. If seasonal products exist, they are a small group hidden inside the averages.

### Zero days over time, one department below the other

In [ ]:
# Zero days over time, one panel per food department (one below the other).
# Each point = one month: % of live item-days with no sale. Thick = all stores, thin = each state.
# 2011 starts 29 Jan: the first months look low only because few items count as "live" yet (grey shading).
def as_timeline(ym):
    s = ym.stack()
    s.index = pd.to_datetime([f'{y}-{MONTHS.index(m) + 1:02d}-01' for y, m in s.index])
    return s.sort_index()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.07,
                    subplot_titles=[f'<b>{d}</b>' for d in DEPTS])
for r, d in enumerate(DEPTS, start=1):
    for k, st in enumerate(STATES):
        s = as_timeline(dept_ym[f'{d}, {st}'])
        fig.add_trace(go.Scatter(x=s.index, y=s, mode='lines', name=st, legendgroup=st, showlegend=(r == 1),
                                 line=dict(color=STATE_COLOR[st], width=1.3, dash=['solid', 'dash', 'dot'][k]),
                                 hovertemplate=f'{d}, {st}<br>%{{x|%b %Y}}: %{{y:.1f}}% no-sale days<extra></extra>'),
                      row=r, col=1)
    s = as_timeline(dept_ym[d])
    fig.add_trace(go.Scatter(x=s.index, y=s, mode='lines', name='All stores', legendgroup='all', showlegend=(r == 1),
                             line=dict(color='#222', width=3),
                             hovertemplate=f'{d}, all stores<br>%{{x|%b %Y}}: %{{y:.1f}}% no-sale days<extra></extra>'),
                  row=r, col=1)
    fig.add_vrect(x0='2011-01-01', x1='2011-06-30', fillcolor='#eee', opacity=0.7, line_width=0, row=r, col=1)
fig.update_yaxes(range=[0, 80], title_text='% no-sale days')
fig.update_layout(title='<b>Zero days over time, per food department</b><br>'
                        '<span style="font-size:13px;color:#444">One point per month, counted from each item\'s first sale, '
                        '25 Dec left out. Grey: first months of 2011, too few live items to trust.</span>',
                  template='simple_white', height=900, hovermode='x unified',
                  margin=dict(t=120, b=90),
                  legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center', yanchor='top'))

fig.show()

### Step 4: item-level seasonality, Food dept 1
Are there "Diwali diya" items: products that sell mostly in one season and sit at zero the rest of the year?
- **Seasonality score:** share of an item's yearly sales in its best 3 months in a row. 25% = perfectly flat; a diya-type item would be 80%+.
- **Weakest-year repeat:** the same 3 months' share in the item's weakest year. A real seasonal item scores high every year.
- **Noise baseline:** the same score after shuffling the item's days at random, to see how high it gets by pure chance.
- Per item, all 10 stores added together. Only items that first sold before 2012 (149 of 216), so each has 4 full years (2012-2015).

In [ ]:
# Step 4: item-level seasonality, Food dept 1 only (for now).
# Per item, all 10 stores added together. Only items that first sold before 2012, so every item has 4 full years (2012-2015).
# Seasonality score = share of the item's yearly sales that falls in its best 3 months in a row.
#   25% = perfectly flat (3 of 12 months). A "Diwali diya" item would be 80-100%.
# Repeat score = the same 3 months' share in the item's WEAKEST year. High only if the peak comes back every year.
# Noise baseline = the same score after shuffling each item's days at random. Shows how high the score gets by pure chance.
DEPT = 'Food dept 1'
rng = np.random.default_rng(0)
item_units = sales[(sales.dept == DEPT).to_numpy()].groupby('item_id')[days].sum()
item_first = pd.DatetimeIndex(dates[(item_units.to_numpy() > 0).argmax(axis=1)])
item_units = item_units[item_first < '2012-01-01']
in_years = day_idx.year.isin(FULL_YEARS)
U = item_units.to_numpy()[:, in_years].astype(float)          # items x days (2012-2015)
yr, mo = day_idx.year[in_years], day_idx.month[in_years]

def month_profile(u):
    """Units per day in each (year, month), then the 12-month profile averaged over the 4 years."""
    ypm = pd.DataFrame(u.T).groupby([yr, mo]).mean()              # (year, month) x items
    return ypm, ypm.groupby(level=1).mean()

def best3(share):
    """For each item: best 3-month window (wrapping Dec->Jan): its start month and its share."""
    s = share.to_numpy()
    win = np.stack([s[[i, (i + 1) % 12, (i + 2) % 12]].sum(0) for i in range(12)])   # 12 windows x items
    return win.argmax(0), win.max(0)

ypm, prof = month_profile(U)
start, score = best3(prof / prof.sum())
repeat = []
for j in range(len(item_units)):
    idx = [(start[j] + k) % 12 for k in range(3)]
    shares = [(ypm.loc[y].iloc[:, j] / ypm.loc[y].iloc[:, j].sum()).iloc[idx].sum() for y in FULL_YEARS]
    repeat.append(min(shares))

U_shuf = rng.permuted(U, axis=1)                                # same days, random order
_, prof_shuf = month_profile(U_shuf)
_, score_shuf = best3(prof_shuf / prof_shuf.sum())

seas = pd.DataFrame({'item': item_units.index.str.split('_').str[-1], 'units a day': U.mean(1),
                     'best 3 months start': [MONTHS[s] for s in start], 'seasonality %': score * 100,
                     'weakest-year repeat %': np.array(repeat) * 100, 'noise score %': score_shuf * 100},
                    index=item_units.index).sort_values('seasonality %', ascending=False)
print(f'{DEPT}: {len(seas)} of {(sales.dept == DEPT).sum() // 10} items have 4 full years')
seas.head(10).round(1)

In [ ]:
# Chart: how seasonal are the items, compared with pure chance?
bins = dict(start=20, end=100, size=2)
fig = go.Figure()
fig.add_trace(go.Histogram(x=seas['noise score %'], xbins=bins, name='Shuffled at random (pure chance)',
                           marker=dict(color='#bbb', line=dict(color='#888', width=1)), opacity=0.8,
                           hovertemplate='score %{x}%: %{y} items (random)<extra></extra>'))
fig.add_trace(go.Histogram(x=seas['seasonality %'], xbins=bins, name=f'Real {DEPT} items',
                           marker=dict(color=plots.PALETTE[3], line=dict(color='white', width=1)), opacity=0.75,
                           hovertemplate='score %{x}%: %{y} items (real)<extra></extra>'))
fig.add_vline(x=25, line=dict(color='#333', dash='dot'), annotation_text='25% = perfectly flat', annotation_position='top right')
fig.add_vrect(x0=80, x1=100, fillcolor=plots.PALETTE[1], opacity=0.12, line_width=0,
              annotation_text='"high seasonality"zone:<br>80%+ of sales in 3 months', annotation_position='top left')
fig.update_layout(title=f'<b>{DEPT}: no item sells mostly in one season</b><br>'
                        '<span style="font-size:13px;color:#444">Score = share of a year\'s sales in the item\'s best 3 months '
                        'in a row (2012-2015, all 10 stores). Grey = same items with their days shuffled at random.</span>',
                  xaxis_title='share of yearly sales in the best 3 months (%)', yaxis_title='number of items',
                  barmode='overlay', template='simple_white', height=480, margin=dict(t=110),
                  legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'))
fig.show()
bands = pd.cut(seas['seasonality %'], [0, 30, 35, 40, 50, 80, 100]).value_counts().sort_index()
pd.DataFrame({'real items': bands,
              'random items': pd.cut(seas['noise score %'], [0, 30, 35, 40, 50, 80, 100]).value_counts().sort_index()})

**Draft observations (check against the chart)**
- **No diya-type items in Food dept 1.** The highest score is 45% (item 011). Nothing comes near the 80% zone, and no item has even half its sales in its best 3 months.
- **But a mild season is real for many items.** Shuffled at random, no item scores above 29%. In the real data, 93 of 149 items score higher than that. So many items lean towards some months, just not strongly.
- **Most "seasonal" items sit between 30% and 45%:** about 1.2 to 1.8 times their fair 3-month share. That is a lean, not an on/off product.

#### Same chart for Food dept 2 and Food dept 3

In [ ]:
# Same seasonality chart for Food dept 2 and Food dept 3 (same score, same noise baseline, same rules as Food dept 1).
# Computed inside a function so the Food dept 1 variables above (seas, ypm, item_units) are not overwritten.
def seasonality_scores(dept, seed=0):
    units = sales[(sales.dept == dept).to_numpy()].groupby('item_id')[days].sum()
    first_day = pd.DatetimeIndex(dates[(units.to_numpy() > 0).argmax(axis=1)])
    units = units[first_day < '2012-01-01']                       # 4 full years 2012-2015
    u = units.to_numpy()[:, in_years].astype(float)
    def score_of(arr):
        prof = pd.DataFrame(arr.T).groupby([yr, mo]).mean().groupby(level=1).mean()
        return best3(prof / prof.sum())[1] * 100
    shuffled = np.random.default_rng(seed).permuted(u, axis=1)
    return pd.DataFrame({'seasonality %': score_of(u), 'noise score %': score_of(shuffled)}, index=units.index)

OTHER_DEPTS = ['Food dept 2', 'Food dept 3']
seas_other = {d: seasonality_scores(d) for d in OTHER_DEPTS}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.14,
                    subplot_titles=[f'<b>{d}</b>: {len(s)} items with 4 full years' for d, s in seas_other.items()])
for r, (d, s) in enumerate(seas_other.items(), start=1):
    fig.add_trace(go.Histogram(x=s['noise score %'], xbins=bins, name='Shuffled at random (pure chance)',
                               legendgroup='noise', showlegend=(r == 1),
                               marker=dict(color='#bbb', line=dict(color='#888', width=1)), opacity=0.8,
                               hovertemplate='score %{x}%: %{y} items (random)<extra></extra>'), row=r, col=1)
    fig.add_trace(go.Histogram(x=s['seasonality %'], xbins=bins, name='Real items', legendgroup='real', showlegend=(r == 1),
                               marker=dict(color=DEPT_COLOR[d], line=dict(color='white', width=1)), opacity=0.75,
                               hovertemplate=f'{d}<br>score %{{x}}%: %{{y}} items (real)<extra></extra>'), row=r, col=1)
    fig.add_vline(x=25, line=dict(color='#333', dash='dot'), row=r, col=1,
                  annotation_text='25% = perfectly flat', annotation_position='top right')
    fig.add_vrect(x0=80, x1=100, fillcolor=plots.PALETTE[1], opacity=0.12, line_width=0, row=r, col=1,
                  annotation_text='"high seasonality" zone:<br>80%+ of sales in 3 months', annotation_position='top left')
    fig.update_yaxes(title_text='number of items', row=r, col=1)
fig.update_xaxes(title_text='share of yearly sales in the best 3 months (%)', row=2, col=1)
fig.update_layout(title='<b>Food dept 2 and 3: no item sells mostly in one season either</b><br>'
                        '<span style="font-size:13px;color:#444">Score = share of a year\'s sales in the item\'s best 3 months '
                        'in a row (2012-2015, all 10 stores). Grey = same items with their days shuffled at random.</span>',
                  barmode='overlay', template='simple_white', height=720, margin=dict(t=110, b=110),
                  legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center', yanchor='top'))
fig.show()
cuts = [0, 30, 35, 40, 50, 80, 100]
pd.concat({d: pd.DataFrame({'real items': pd.cut(s['seasonality %'], cuts).value_counts().sort_index(),
                            'random items': pd.cut(s['noise score %'], cuts).value_counts().sort_index()})
           for d, s in seas_other.items()}, axis=1)

**Draft observations (check against the chart)**
- **Same picture as Food dept 1.** Most items score 25-40% (median about 29%). Shuffled at random, nothing goes above 30%, so a mild lean to some months is real for many items (156 of 245 in Food dept 2, 197 of 498 in Food dept 3), but it is only a lean.
- **Exactly one real "holiday item" in each department:**
  - **Food dept 2, item 255 (65%):** about 7 a day most of the year, then about 100 a day every **November** (2013-2015) and about 30 a day in December. A Thanksgiving product.
  - **Food dept 3, item 069 (64%):** about 5-15 a day most of the year, then 85-127 a day every **December**, about 30 in November. A Christmas product.
- **Even these two never drop to zero off-season:** they still sell 5-10 a day across all stores. So they are "holiday booster" items, not "only in season" diya items.
- Next highest: 3 items in Food dept 2 and 16 in Food dept 3 score 40%+; everything else is below 50%.
- **Phase 1 answer (draft, all 3 departments):** seasonal products do **not** explain the 40-50% no-sale days. Only 2 of about 890 items are strongly seasonal, and even they keep selling all year.

In [ ]:
# The 6 most seasonal items: does the peak come back in the same months every year? One line per year.
top = seas.head(6)
fig = make_subplots(rows=2, cols=3, shared_xaxes=True, horizontal_spacing=0.06, vertical_spacing=0.18,
                    subplot_titles=[f"<b>item {r['item']}</b> · {r['units a day']:.0f} a day<br>"
                                    f"<span style='font-size:11px'>best 3 months from {r['best 3 months start']}: "
                                    f"{r['seasonality %']:.0f}% (weakest year {r['weakest-year repeat %']:.0f}%)</span>"
                                    for _, r in top.iterrows()])
col_of = {item: j for j, item in enumerate(item_units.index)}
for k, (item_id, r) in enumerate(top.iterrows()):
    for n, y in enumerate(FULL_YEARS):
        v = ypm.loc[y].iloc[:, col_of[item_id]]
        fig.add_trace(go.Scatter(x=MONTHS, y=v.values, mode='lines', name=str(y), legendgroup=str(y),
                                 showlegend=(k == 0), line=dict(color=plots.PALETTE[n], width=1.8,
                                                                dash=['solid', 'dash', 'dot', 'dashdot'][n]),
                                 hovertemplate=f"item {r['item']}, %{{x}} {y}: %{{y:.1f}} a day<extra></extra>"),
                      row=k // 3 + 1, col=k % 3 + 1)
fig.update_yaxes(rangemode='tozero')
fig.update_yaxes(title_text='units a day', col=1)
fig.update_layout(title=f'<b>{DEPT}: the 6 "most seasonal" items, one line per year</b><br>'
                        '<span style="font-size:13px;color:#444">A real seasonal item peaks in the same months every year. '
                        'All 10 stores added together.</span>',
                  template='simple_white', height=620, margin=dict(t=120),
                  legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'))
fig.show()

**Draft observations**
- **Only some peaks come back every year.** Item 116 (July peak) keeps about 41% of its sales in those months even in its weakest year, so it is genuinely seasonal. Items 011 and 075 drop to about 30% in their weakest year, so part of their score is a one-off year.
- The best 3-month windows mostly start in **July, October or December** (so Jul-Sep, Oct-Dec, Dec-Feb): late summer and the holiday season.
- Only 14 of 149 items keep 30%+ in the same 3 months in every year.

In [ ]:
# Link back to zero days: do the more seasonal items have more no-sale days (per item, per store)?
item_store = sales.loc[(sales.dept == DEPT).to_numpy() & sales.item_id.isin(seas.index).to_numpy(),
                       ['item_id', 'store', 'zero_share', 'mean_daily']]
item_store['seasonality band'] = pd.cut(item_store.item_id.map(seas['seasonality %']), [0, 30, 35, 40, 100],
                                        labels=['flat (under 30%)', '30-35%', '35-40%', 'most seasonal (40%+)'])
zero_by_band = item_store.groupby('seasonality band', observed=True).agg(
    items=('item_id', 'nunique'), item_store_series=('item_id', 'size'),
    avg_no_sale_days_pct=('zero_share', 'mean'), avg_units_a_day=('mean_daily', 'mean'))
zero_by_band.round(1)

**Draft observations**
- **More seasonal items do have more no-sale days:** about 66% for the 8 most seasonal items vs 51% for the flat ones.
- **But they also sell about half as much** (0.9 a day per store vs 1.9). Slow items have more empty days anyway, so this table cannot say whether the season or the slow sales causes the empty days.
- **Size of the effect:** the most seasonal band is 8 items (80 of 1,490 item-store series, about 5%). Even if all their empty days were seasonal, it would explain only a small part of Food dept 1's zero days.
- **Phase 1 answer for Food dept 1 (draft):** seasonal products do **not** explain the 40-50% no-sale days. The empty days are mainly a slow-selling (and possibly stock-out) issue.

## What to decide here
*(fill in at the end of the day)*